# SpaceX Launch Records Dashboard (Plotly Dash)

IBM Applied Data Science Capstone — Interactive Visual Analytics.

This notebook completes **TASK 1–4** of *Build a Dashboard Application with Plotly Dash*.
The same code is in `spacex_dash_app.py`. Run the app with:

```bash
python spacex_dash_app.py
```

Then open http://127.0.0.1:8050


In [ ]:
import pandas as pd
import dash
from dash import html, dcc
from dash.dependencies import Input, Output
import plotly.express as px

spacex_df = pd.read_csv("spacex_launch_dash.csv")
max_payload = spacex_df["Payload Mass (kg)"].max()
min_payload = spacex_df["Payload Mass (kg)"].min()
spacex_df.head()

## TASK 1: Launch Site dropdown

`id='site-dropdown'`, default `ALL`, searchable, one option per launch site.

## TASK 2: Success pie chart callback

- ALL sites: pie of total successful launches by site (`class` summed per site)
- One site: Success vs Failure counts

## TASK 3: Payload range slider

`id='payload-slider'`, min 0, max 10000, step 1000, value `[min_payload, max_payload]`.

## TASK 4: Payload vs class scatter callback

Inputs: site dropdown + payload slider. Color = `Booster Version Category`.

In [ ]:
app = dash.Dash(__name__)

app.layout = html.Div(children=[
    html.H1(
        "SpaceX Launch Records Dashboard",
        style={"textAlign": "center", "color": "#503D36", "font-size": 40},
    ),
    # TASK 1
    dcc.Dropdown(
        id="site-dropdown",
        options=[
            {"label": "All Sites", "value": "ALL"},
            {"label": "CCAFS LC-40", "value": "CCAFS LC-40"},
            {"label": "VAFB SLC-4E", "value": "VAFB SLC-4E"},
            {"label": "KSC LC-39A", "value": "KSC LC-39A"},
            {"label": "CCAFS SLC-40", "value": "CCAFS SLC-40"},
        ],
        value="ALL",
        placeholder="Select a Launch Site here",
        searchable=True,
    ),
    html.Br(),
    # TASK 2 graph placeholder
    html.Div(dcc.Graph(id="success-pie-chart")),
    html.Br(),
    html.P("Payload range (Kg):"),
    # TASK 3
    dcc.RangeSlider(
        id="payload-slider",
        min=0,
        max=10000,
        step=1000,
        marks={0: "0", 2500: "2500", 5000: "5000", 7500: "7500", 10000: "10000"},
        value=[min_payload, max_payload],
    ),
    # TASK 4 graph placeholder
    html.Div(dcc.Graph(id="success-payload-scatter-chart")),
])


@app.callback(
    Output(component_id="success-pie-chart", component_property="figure"),
    Input(component_id="site-dropdown", component_property="value"),
)
def get_pie_chart(entered_site):
    if entered_site == "ALL":
        return px.pie(
            spacex_df,
            values="class",
            names="Launch Site",
            title="Total Success Launches By Site",
        )
    filtered_df = spacex_df[spacex_df["Launch Site"] == entered_site]
    site_counts = filtered_df["class"].value_counts().reset_index()
    site_counts.columns = ["class", "count"]
    site_counts["class"] = site_counts["class"].map({1: "Success", 0: "Failure"})
    return px.pie(
        site_counts,
        values="count",
        names="class",
        title=f"Total Success vs Failure for site {entered_site}",
    )


@app.callback(
    Output(component_id="success-payload-scatter-chart", component_property="figure"),
    [
        Input(component_id="site-dropdown", component_property="value"),
        Input(component_id="payload-slider", component_property="value"),
    ],
)
def get_scatter_chart(entered_site, payload_range):
    low, high = payload_range
    filtered_df = spacex_df[
        (spacex_df["Payload Mass (kg)"] >= low)
        & (spacex_df["Payload Mass (kg)"] <= high)
    ]
    if entered_site == "ALL":
        return px.scatter(
            filtered_df,
            x="Payload Mass (kg)",
            y="class",
            color="Booster Version Category",
            title="Correlation between Payload and Success for all Sites",
        )
    filtered_df = filtered_df[filtered_df["Launch Site"] == entered_site]
    return px.scatter(
        filtered_df,
        x="Payload Mass (kg)",
        y="class",
        color="Booster Version Category",
        title=f"Correlation between Payload and Success for site {entered_site}",
    )

Run the dashboard (opens on port 8050). Stop the cell when finished.

In [ ]:
if __name__ == "__main__":
    app.run(debug=False)